# 04 – B1: retrieval-score baseline (Kaggle)

Baseline B1: use the cosine similarity between the query and the retrieved passage (multilingual-e5-large, with `query:`/`passage:` prefixes) as the only confidence signal. Same 960 instances as the d1 gate run. Output: `retrieval_score_mE5large.csv` → copy to `results/baselines/`. Analysis: `scripts/analyze_baselines.py`.

Kaggle: GPU on, Internet on, add the repo `data/dev/AISIS_DevSet_v1.xlsx` and `data/gate/gate_questions_v2.json` as a dataset. First cell: `!pip install -q sentence-transformers`.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import glob, json, os, time
import pandas as pd

xlsx = glob.glob("/kaggle/input/**/AISIS_DevSet_v1.xlsx", recursive=True)[0]
qjson = glob.glob("/kaggle/input/**/gate_questions_v2.json", recursive=True)[0]
dev = pd.read_excel(xlsx, sheet_name="DevSet").fillna("")
QUESTIONS = json.load(open(qjson, encoding="utf-8"))["questions"]
TOOLS = {k: v for k, v in QUESTIONS["tool"]["criteria"].items() if k != "none"}

# identical instance construction to notebook 01 (d1 gate): standard + dialect (+ wrong_passage for answerable)
instances = []
for r in dev.to_dict("records"):
    base = dict(dev_id=r["dev_id"], type=r["type"], topic=r["topic"],
                gold_action=r["label_action"], gold_supported=r["label_supported"],
                gold_specific=r["label_specific"], gold_emergency=r["label_emergency"],
                gold_tool=r["label_tool"])
    instances.append({**base, "version": "standard", "query": r["query_standard"], "passage": r["passage_text"]})
    instances.append({**base, "version": "dialect", "query": r["query_dialect_synth"], "passage": r["passage_text"]})
    if r["type"] == "answerable":
        instances.append({**base, "version": "wrong_passage", "query": r["query_standard"],
                          "passage": r["wrong_passage_text"],
                          "gold_action": "cannot_answer", "gold_supported": "not_supported"})
print(len(instances), "instances")  # expected 960

In [ ]:
import numpy as np, torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
q_emb = model.encode(["query: " + i["query"] for i in instances], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
p_emb = model.encode(["passage: " + i["passage"] for i in instances], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
sim = (q_emb * p_emb).sum(1)   # cosine(query, retrieved passage) = the "retrieval score"

out = pd.DataFrame({"dev_id": [i["dev_id"] for i in instances],
                    "version": [i["version"] for i in instances],
                    "sim": sim})
out.to_csv("/kaggle/working/retrieval_score_mE5large.csv", index=False)
print(out.groupby("version").sim.describe().round(3))